In [ ]:
# Import libraries
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString
from typing import Union, List


# Function to normalize longitude values
def normalize_longitude(lon: Union[float, List[float], np.ndarray]) -> Union[float, np.ndarray]:
    """
    Normalize longitude values so they fall between -180 and 180.

    Parameters
    ----------
    lon : float or array-like
        Longitude value(s). If values are greater than 180, they are converted.

    Returns
    -------
    float or numpy.ndarray
        Longitude value(s) corrected to the range [-180, 180]
    """

    lon_array = np.array(lon)
    lon_array = np.where(lon_array > 180, lon_array - 360, lon_array)

    if lon_array.size == 1:
        return float(lon_array)

    return lon_array


# Function to create geometry for GeoPandas
def create_geometry(
    lat: Union[float, List[float], np.ndarray],
    lon: Union[float, List[float], np.ndarray],
    name: str,
    description: str
) -> gpd.GeoDataFrame:
    """
    Create a GeoDataFrame containing either a Point or LineString.

    Parameters
    ----------
    lat : float or array-like
        Latitude value(s)
    lon : float or array-like
        Longitude value(s)
    name : str
        Name of the feature
    description : str
        Description of the feature

    Returns
    -------
    geopandas.GeoDataFrame
        GeoDataFrame with geometry and attributes
    """

    lon = normalize_longitude(lon)

    # Determine if Point or LineString
    if isinstance(lat, (list, np.ndarray)) and len(lat) > 1:
        coords = list(zip(lon, lat))
        geometry = LineString(coords)
    else:
        geometry = Point(lon, lat)

    gdf = gpd.GeoDataFrame(
        {
            "name": [name],
            "description": [description],
            "geometry": [geometry]
        },
        crs="EPSG:4326"
    )

    return gdf


# ----------------------------------------------------
# Load the survey CSV file
# ----------------------------------------------------

survey = pd.read_csv(r"C:\\Users\\jaely\\OneDrive\\Documents\\AES408\\bradford_lls.csv")

latitudes = survey["lat"].values
longitudes = survey["lon"].values


# ----------------------------------------------------
# Create GeoDataFrame using the function
# ----------------------------------------------------

greenway_gdf = create_geometry(
    latitudes,
    longitudes,
    name="Bradford Creek Greenway",
    description="Survey points for proposed greenway repaving in Madison, AL"
)


# ----------------------------------------------------
# Plot the data
# ----------------------------------------------------

fig, ax = plt.subplots(figsize=(8,6))

greenway_gdf.plot(ax=ax)

ax.set_title("Bradford Creek Greenway Repaving Survey")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()


# ----------------------------------------------------
# Save the shapefile
# ----------------------------------------------------

greenway_gdf.to_file("bradford_creek_greenway.shp")

print("Shapefile saved successfully.")